# PCS1: inherit -> new evidence -> new deduction

**Question.** PCS (portable cognitive state) needs a receiver B to *continue* from a state computed by A, not just answer the same question. Here A = Qwen3-4B reads a short variable-update program P (the fused section). B = Qwen3-0.6B receives A's KV cache for P through the released, frozen general C2C fuser (`nics-efc/C2C_Fuser/qwen3_0.6b+qwen3_4b_Fuser`), then reads *new evidence* U (an operation on one of P's variables) that only B sees (unfused tail), and must output a single digit: the new deduction. Readout: logits over the 10 digit tokens after "The answer is ". No chain of thought.

**Tests.** (1) new-deduction gain: acc(C) - mean over K=3 wrong-P caches, and vs a calibrated receiver; (2) fidelity: when A's own belief about the variable is wrong, does B's answer follow the U-answer implied by A's belief, beyond what wrong-P caches give; (3) bridge vs a text handoff of A's value.

The fuser was trained for one-shot QA on identical inputs, not for continuation with new evidence; this tests whether an existing bridge supports PCS-style continuation. **Read `PREREG_pcs1.md` first.** `MODE = "pilot"` runs only R, S_full, S_state (no fuser, no C) to choose depths; `MODE = "main"` runs everything.

Run on Kaggle: GPU T4, Internet on, Run All.

In [ ]:
# --- 1. Config ---
import os, sys, json, time, math, random, re, subprocess, platform, tempfile, hashlib, datetime, gc
from pathlib import Path

def _flag(name):
    return os.environ.get(name, "").lower() in ("1", "true", "yes")

SMOKE = _flag("PCS_SMOKE")   # CPU rehearsal with tiny RANDOM models (author's validation only)
MODE = "main"                # "pilot" (R, S_full, S_state, R_state only) or "main" (everything)

CFG = dict(
    SEED=0,
    PILOT_DEPTHS=[0, 1, 2, 3], PILOT_N=200,
    MAIN_DEPTHS=[1], MAIN_N_PER_DEPTH=3000,       # frozen in PREREG_pcs1.md after the pilot
    K_MM=3, N_BOOT=10000, CAL_SEED=0, PIN_TORCH=False,
    MAIN_CAP_MIN=120,           # item loop stops (logged) if it runs past this many minutes; items are processed in a fixed shuffled order
    GATE_PROBES_CPU=12, GATE_CPU_CAP_SEC=420, GATE_MIN_REF=8,
)
if SMOKE:
    CFG.update(PILOT_DEPTHS=[1, 2], PILOT_N=6, MAIN_DEPTHS=[1, 2], MAIN_N_PER_DEPTH=6, N_BOOT=300, GATE_PROBES_CPU=6, GATE_MIN_REF=1)
if MODE == "pilot":
    CFG["DEPTHS"], CFG["N_PER_DEPTH"] = CFG["PILOT_DEPTHS"], CFG["PILOT_N"]
else:
    CFG["DEPTHS"], CFG["N_PER_DEPTH"] = CFG["MAIN_DEPTHS"], CFG["MAIN_N_PER_DEPTH"]
SEED = CFG["SEED"]
IS_KAGGLE = Path("/kaggle/working").exists()
OUT_DIR = Path(os.environ.get("PCS_OUT", "/kaggle/working" if IS_KAGGLE else str(Path.cwd())))
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH, PARTIAL_PATH = OUT_DIR / "results.json", OUT_DIR / "results_partial.json"

RECV_ID, SHAR_ID = "Qwen/Qwen3-0.6B", "Qwen/Qwen3-4B"
FUSER_REPO, FUSER_SUB = "nics-efc/C2C_Fuser", "qwen3_0.6b+qwen3_4b_Fuser"
FP32_GB = 20.6   # fp32 stack (both models + fuser), decides whether the fp16 gate's fp32 reference can run on the GPU
SMOKE_RECV = dict(hidden_size=128, intermediate_size=256, num_attention_heads=16, num_key_value_heads=8, head_dim=128, num_hidden_layers=28)
SMOKE_SHAR = dict(hidden_size=128, intermediate_size=256, num_attention_heads=16, num_key_value_heads=8, head_dim=128, num_hidden_layers=36)
C2C_URL = "https://github.com/thu-nics/C2C"
C2C_COMMIT = "3ca0e98020bac1f36000623afcb8dfc6f8c29bdf"
PINS = {"transformers": "4.52.4", "tokenizers": "0.21.4", "huggingface_hub": "0.34.3"}
DEVIATIONS = []
T0 = time.time()
print(f"mode: {MODE}{' (SMOKE)' if SMOKE else ''} | depths {CFG['DEPTHS']} x {CFG['N_PER_DEPTH']} | {RECV_ID} <- {SHAR_ID} | out={OUT_DIR}")

In [ ]:
# --- 2. Install pinned deps, GPU sanity (runs BEFORE torch is imported) ---
def sh(cmd, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check)

def gpu_query():
    try:
        o = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=20)
        if o.returncode == 0 and o.stdout.strip():
            name, cc, mem = [x.strip() for x in o.stdout.strip().splitlines()[0].split(",")]
            return name, float(cc), mem
    except Exception:
        pass
    return None

def installed(pkg):
    import importlib.metadata as md
    try:
        return md.version(pkg)
    except Exception:
        return None

if "torch" in sys.modules and not SMOKE:
    print("note: torch is already imported in this kernel; if installs below change anything, restart and Run All")

if not SMOKE:
    g = gpu_query()
    if g is None:
        raise RuntimeError("No NVIDIA GPU found. Kaggle: Settings > Accelerator > GPU T4. Colab: Runtime > Change runtime type > T4 GPU.")
    print("GPU:", g)
    if (g[1] < 7.0 or CFG["PIN_TORCH"]) and "torch" not in sys.modules:
        # e.g. Kaggle P100 (sm_60): recent default torch wheels may lack sm_60 kernels. C2C itself pins torch 2.6.0.
        sh([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
    need = [f"{k}=={v}" for k, v in PINS.items() if installed(k) != v]
    if need:
        sh([sys.executable, "-m", "pip", "install", "-q", *need])
    for pkg in ("datasets", "pandas", "numpy", "tqdm"):
        if installed(pkg) is None:
            sh([sys.executable, "-m", "pip", "install", "-q", pkg])

for k, v in PINS.items():
    got = installed(k)
    if got != v:
        raise RuntimeError(f"{k}=={got}, expected {v}. Restart the kernel and Run All (the install cell must run before anything imports {k}).")
print({k: installed(k) for k in list(PINS) + ["datasets", "numpy"]})

In [ ]:
# --- 3. Clone C2C at the audited commit ---
C2C_DIR = Path(os.environ.get("PCS_C2C_DIR", str(Path(tempfile.gettempdir()) / "C2C")))
if not (C2C_DIR / "rosetta").exists():
    C2C_DIR.mkdir(parents=True, exist_ok=True)
    try:
        sh(["git", "init", "-q", str(C2C_DIR)])
        sh(["git", "-C", str(C2C_DIR), "remote", "add", "origin", C2C_URL])
        sh(["git", "-C", str(C2C_DIR), "fetch", "-q", "--depth", "1", "origin", C2C_COMMIT])
        sh(["git", "-C", str(C2C_DIR), "checkout", "-q", "FETCH_HEAD"])
    except Exception as e:
        print("pinned checkout failed (", e, ") -> falling back to default branch HEAD")
        DEVIATIONS.append("C2C pinned commit unavailable; used HEAD of default branch")
        __import__("shutil").rmtree(C2C_DIR, ignore_errors=True)
        sh(["git", "clone", "-q", "--depth", "1", C2C_URL, str(C2C_DIR)])
sys.path.insert(0, str(C2C_DIR))   # `rosetta` is a namespace package: do NOT `pip install -e .` (it would re-pin torch)
head = subprocess.run(["git", "-C", str(C2C_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("C2C commit:", head)

In [ ]:
# --- 4. Imports, device, dtype policy, API presence checks ---
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, GenerationConfig
from huggingface_hub import snapshot_download

from rosetta.model.wrapper import RosettaModel
from rosetta.model.projector import load_projector
from rosetta.utils.evaluate import set_default_chat_template

for _n in ("forward", "generate", "load_projector_config"):
    assert hasattr(RosettaModel, _n), f"C2C API changed: RosettaModel.{_n} missing"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    GPU_NAME = torch.cuda.get_device_name(0)
    CC = torch.cuda.get_device_capability(0)
    NATIVE_BF16 = CC[0] >= 8
    torch.cuda.manual_seed_all(SEED)
else:
    if not SMOKE:
        raise RuntimeError("CUDA not available. Enable a GPU runtime.")
    DEVICE, GPU_NAME, CC, NATIVE_BF16 = torch.device("cpu"), "cpu", (0, 0), False
print(f"torch {torch.__version__} | transformers {installed('transformers')} | device {DEVICE} ({GPU_NAME}, cc {CC}) | native bf16: {NATIVE_BF16}")
LETTERS = "0123456789"          # (name kept for the shared calibration helpers) the ten digit tokens are the answer alphabet
RESP_TEXT = "The answer is "     # the next token is a digit

In [ ]:
# --- 6. Models + fuser (dtype-aware loader; the repo's own loader hard-codes bf16, which a T4 cannot do natively) ---
def fuser_dir():
    pats = [f"{FUSER_SUB}/final/*.json"] if SMOKE else [f"{FUSER_SUB}/final/*", f"{FUSER_SUB}/config.json"]
    root = snapshot_download(repo_id=FUSER_REPO, allow_patterns=pats)
    return os.path.join(root, FUSER_SUB, "final")

FUSER_PATH = fuser_dir() if MODE == "main" else None
if FUSER_PATH:
    print("fuser dir:", FUSER_PATH, "| files:", len(os.listdir(FUSER_PATH)))

def _tiny(cfg_id, **over):
    cfg = AutoConfig.from_pretrained(cfg_id)
    for k, v in over.items():
        setattr(cfg, k, v)
    return cfg

def build_stack(dtype, device=None, with_fuser=True):
    """Receiver + sharer + fuser in `dtype` on `device` (default: the global DEVICE)."""
    device = device if device is not None else DEVICE
    tok_r = AutoTokenizer.from_pretrained(RECV_ID); set_default_chat_template(tok_r, RECV_ID)
    tok_s = AutoTokenizer.from_pretrained(SHAR_ID)
    if SMOKE:
        torch.manual_seed(SEED)
        recv = AutoModelForCausalLM.from_config(_tiny(RECV_ID, **SMOKE_RECV), torch_dtype=dtype)
        shar = AutoModelForCausalLM.from_config(_tiny(SHAR_ID, **SMOKE_SHAR), torch_dtype=dtype)
    else:
        recv = AutoModelForCausalLM.from_pretrained(RECV_ID, torch_dtype=dtype)
        shar = AutoModelForCausalLM.from_pretrained(SHAR_ID, torch_dtype=dtype)
    recv, shar = recv.to(device).eval(), shar.to(device).eval()
    if not with_fuser:
        return dict(tok_r=tok_r, tok_s=tok_s, recv=recv, shar=shar, ros=None, dtype=dtype, n_proj=0)
    n_proj = len([f for f in os.listdir(FUSER_PATH) if re.fullmatch(r"projector_\d+\.json", f)])
    assert n_proj == recv.config.num_hidden_layers, f"fuser has {n_proj} projectors, receiver has {recv.config.num_hidden_layers} layers"
    projs = []
    for t_ in range(n_proj):
        over = {"dtype": dtype}
        if SMOKE:
            over.update(hidden_dim=64, intermediate_dim=64)
        p = load_projector(os.path.join(FUSER_PATH, f"projector_{t_}.json"), override_args=over)
        pt = os.path.join(FUSER_PATH, f"projector_{t_}.pt")
        if not SMOKE:
            res = p.load_state_dict(torch.load(pt, map_location="cpu", weights_only=True), strict=False)   # repo uses strict=False; we check
            if res.missing_keys or res.unexpected_keys:
                raise RuntimeError(f"fuser projector_{t_}: missing={res.missing_keys} unexpected={res.unexpected_keys}")
            bad = [k for k, v in p.state_dict().items() if v.is_floating_point() and not torch.isfinite(v).all()]
            if bad:   # e.g. bf16 -> fp16 overflow while casting the released weights
                raise RuntimeError(f"fuser projector_{t_}: non-finite weights after cast to {dtype}: {bad[:5]}")
        if SMOKE:   # random projectors start with closed gates (logit 0 -> no fusion); open them so the fusion path is exercised
            with torch.no_grad():
                p.key_gate_logit.fill_(1.0); p.value_gate_logit.fill_(1.0)
        projs.append(p.to(device))
    ros = RosettaModel(model_list=[recv, shar], base_model_idx=0, projector_list=projs).to(device).eval()
    ros.load_projector_config(os.path.join(FUSER_PATH, "projector_config.json"))
    assert len(ros.projector_dict[0][1]) == recv.config.num_hidden_layers, "projector map must cover every receiver layer"
    return dict(tok_r=tok_r, tok_s=tok_s, recv=recv, shar=shar, ros=ros, dtype=dtype, n_proj=n_proj)

def use_stack(st):
    global TOK_R, TOK_S, RECV, SHAR, ROS, DTYPE_USED, OPT_IDS, OPT_T, RESP_IDS, EOS_R, EOS_S, PAD_R, PAD_S
    TOK_R, TOK_S, RECV, SHAR, ROS, DTYPE_USED = st["tok_r"], st["tok_s"], st["recv"], st["shar"], st["ros"], st["dtype"]
    d_r = [TOK_R(str(d), add_special_tokens=False)["input_ids"] for d in range(10)]
    d_s = [TOK_S(str(d), add_special_tokens=False)["input_ids"] for d in range(10)]
    assert d_r == d_s and all(len(x) == 1 for x in d_r), "digit tokens must be single tokens and identical in both tokenizers"
    OPT_IDS = [x[0] for x in d_r]
    OPT_T = torch.tensor(OPT_IDS, device=DEVICE)
    r1, r2 = TOK_R(RESP_TEXT, add_special_tokens=False)["input_ids"], TOK_S(RESP_TEXT, add_special_tokens=False)["input_ids"]
    assert r1 == r2, "response-suffix tokenization differs between tokenizers"
    RESP_IDS = r1
    EOS_R, EOS_S = RECV.generation_config.eos_token_id, SHAR.generation_config.eos_token_id
    PAD_R = TOK_R.pad_token_id if TOK_R.pad_token_id is not None else (EOS_R[0] if isinstance(EOS_R, list) else EOS_R)
    PAD_S = TOK_S.pad_token_id if TOK_S.pad_token_id is not None else (EOS_S[0] if isinstance(EOS_S, list) else EOS_S)

def free_stack(st):
    for k in list(st):
        st[k] = None
    for n_ in ("RECV", "SHAR", "ROS"):
        globals()[n_] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# --- 7. Inference primitives (one readout: the ten digit-token logits after "The answer is ") ---
COUNTERS = {"nonfinite": 0}

def encode_user(tok, user_text):
    s = tok.apply_chat_template([{"role": "user", "content": user_text}], tokenize=False,
                                add_generation_prompt=True, enable_thinking=False)
    return tok(s, add_special_tokens=False)["input_ids"]

def to_pred(logits_vec):
    """a = argmax letter (ties -> lower letter); p = softmax over the four letter logits; lp = log-softmax over them (the
    calibrated baselines of PREREG_run2 (a), kept in run 3 use lp)."""
    sel = logits_vec[OPT_T].float()
    if not torch.isfinite(sel).all():
        COUNTERS["nonfinite"] += 1
        return {"a": "?", "p": [None] * 10, "lp": [None] * 10}
    lp = torch.log_softmax(sel, 0).cpu().numpy()
    p = np.exp(lp)
    return {"a": LETTERS[int(lp.argmax())], "p": [round(float(x), 4) for x in p], "lp": [round(float(x), 5) for x in lp]}

@torch.no_grad()
def lm_logits(model, ids, mask=None, positions=None):
    """Plain HF forward (receiver-alone / sharer-alone). Returns last-position logits."""
    x = torch.tensor([ids], device=DEVICE)
    pos = torch.arange(x.shape[1], device=DEVICE).unsqueeze(0) if positions is None else torch.tensor([positions], device=DEVICE)
    am = None if mask is None else torch.tensor([mask], device=DEVICE)
    return model(input_ids=x, attention_mask=am, position_ids=pos, use_cache=False, logits_to_keep=1).logits[0, -1]

def _kv_index(n_instr, n_total, sharer_mask=1):
    a = torch.tensor([sharer_mask, 0], dtype=torch.long).repeat(n_instr, 1).unsqueeze(0).to(DEVICE)
    b = torch.tensor([-1, 0], dtype=torch.long).repeat(n_total - n_instr, 1).unsqueeze(0).to(DEVICE)
    return [a, b]

@torch.no_grad()
def c2c_logits(ids_r, ids_s, n_instr, mask_r=None, mask_s=None, sharer_mask=1, as_tensor=False):
    """C2C-fused forward. ids_r / ids_s: receiver / sharer token ids, same length. The first n_instr positions are
    fused (kv_cache_index [1,0]); the trailing response tokens are not (and the sharer never sees them: C2C only
    computes sharer caches for the fused sections)."""
    assert len(ids_r) == len(ids_s), "C2C needs position-aligned inputs (equal length)"
    n = len(ids_r)
    xr, xs = torch.tensor([ids_r], device=DEVICE), torch.tensor([ids_s], device=DEVICE)
    mr = torch.tensor([mask_r if mask_r is not None else [1] * n], device=DEVICE)
    ms = torch.tensor([mask_s if mask_s is not None else [1] * n], device=DEVICE)
    pos = torch.arange(n, device=DEVICE).unsqueeze(0)
    kv = _kv_index(n_instr, n, sharer_mask)
    if as_tensor:   # exactly the call pattern of C2C's own evaluator for same-tokenizer pairs
        out = ROS.forward(kv_cache_index=kv, input_ids=xr, attention_mask=mr, position_ids=pos)
    else:
        out = ROS.forward(kv_cache_index=kv, input_ids=[xr, xs], attention_mask=[mr, ms], position_ids=pos, use_cache=True)
    return out.logits[0, -1]

In [ ]:
# --- 8. Statistics helpers (bootstrap over items) ---
def boot_ci(x, n_boot=None, seed=0):
    """Percentile bootstrap CI of the mean. Pass per-item differences for paired comparisons."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n == 0:
        return (float("nan"),) * 3
    n_boot = n_boot or CFG["N_BOOT"]
    rng = np.random.default_rng(seed)
    chunk = max(1, int(2_000_000 // n))
    means, done = [], 0
    while done < n_boot:
        m = min(chunk, n_boot - done)
        means.append(x[rng.integers(0, n, size=(m, n))].mean(1))
        done += m
    means = np.concatenate(means)
    return float(x.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

def boot_ci_unpaired_diff(xa, xb, n_boot=None, seed=0):
    xa, xb = np.asarray(xa, float), np.asarray(xb, float)
    if len(xa) == 0 or len(xb) == 0:
        return (float("nan"),) * 3
    n_boot = n_boot or CFG["N_BOOT"]
    rng = np.random.default_rng(seed)
    da = np.array([xa[rng.integers(0, len(xa), len(xa))].mean() for _ in range(n_boot)])
    db = np.array([xb[rng.integers(0, len(xb), len(xb))].mean() for _ in range(n_boot)])
    d = da - db
    return float(xa.mean() - xb.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def fmt(ci, pct=False):
    m, lo, hi = ci
    if m != m:
        return "n/a"
    return f"{m*100:5.1f}% [{lo*100:5.1f},{hi*100:5.1f}]" if pct else f"{m:+.3f} [{lo:+.3f},{hi:+.3f}]"

def table(rows, header):
    if not rows:
        print(*header); return
    w = [max(len(str(h)), *(len(str(r[i])) for r in rows)) for i, h in enumerate(header)]
    line = "  ".join(str(h).ljust(w[i]) for i, h in enumerate(header))
    print(line); print("-" * len(line))
    for r in rows:
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(r)))

In [ ]:
# --- 9. Task: variable-update program P (fused section), new evidence U (unfused tail), exact single-digit answers ---
RECORDS, ITEMS = {}, {}
HEADER = "Track the variables below. All arithmetic is mod 10.\n"
VARS = "abc"
K_INVERTIBLE = (1,)   # addition-only (v2): U is  e = (x + m) % 10  with m in 1..9, so a wrong belief about x maps to a different U-answer (bijection)

def gen_program(rng, depth):
    """Three initial assignments, then `depth` update steps (depth 0 = none: a stored-value read). Step t is ADDITION-ONLY:  x = (y + z) % 10  or
    x = (y + c) % 10, where y is the variable updated at step t-1 (a dependency chain, so depth is real), z a random variable, c a constant 1..9.
    Returns lines, final values, last-updated (queried) variable."""
    vals = {v: rng.randrange(10) for v in VARS}
    lines = [f"{v} = {vals[v]}" for v in VARS]
    prev = rng.choice(VARS)
    last = prev
    for _ in range(depth):
        x = rng.choice(VARS)
        if rng.random() < 0.5:
            z = rng.choice(VARS); vals[x] = (vals[prev] + vals[z]) % 10; lines.append(f"{x} = ({prev} + {z}) % 10")
        else:
            c = rng.randrange(1, 10); vals[x] = (vals[prev] + c) % 10; lines.append(f"{x} = ({prev} + {c}) % 10")
        prev = last = x
    return lines, vals, last

def chat_split(tok):
    s = tok.apply_chat_template([{"role": "user", "content": "@@X@@"}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    pre, post = s.split("@@X@@")
    return pre, post

def enc(tok, text):
    return tok(text, add_special_tokens=False)["input_ids"]

def make_item(depth, idx, salt, with_partners=True):
    """One item. U-answer constraint: it differs from every variable's final value in P, so B cannot answer by copying a state value."""
    rng = random.Random(f"{SEED}:{salt}:{depth}:{idx}")
    while True:
        lines, vals, xvar = gen_program(rng, depth)
        k, m = 1, rng.randrange(1, 10)
        ans = (vals[xvar] * k + m) % 10
        if ans not in vals.values():
            break
    pre, post = chat_split(TOK_R)
    fused_text = HEADER + "\n".join(lines) + "\n"
    p1 = enc(TOK_R, pre + fused_text)                                           # the fused section: chat prefix + header + program P
    resp = "The answer is "
    U_text = f"Now e = ({xvar} + {m}) % 10. What is e? Answer with a single digit."
    Q_text = f"What is {xvar} now? Answer with a single digit."
    it = dict(id=f"d{depth}:{idx}", depth=depth, idx=idx, salt=salt, lines=lines, xvar=xvar, X=vals[xvar], k=k, m=m, ans=ans, vals=dict(vals),
              pre=pre, post=post, fused_text=fused_text, U_text=U_text, Q_text=Q_text, resp=resp,
              p1=p1, n_p1=len(p1), p2U=enc(TOK_R, U_text + post + resp), p2Q=enc(TOK_R, Q_text + post + resp))
    it["ids_full"], it["ids_state"] = p1 + it["p2U"], p1 + it["p2Q"]
    assert it["ids_full"] == enc(TOK_S, pre + fused_text) + enc(TOK_S, U_text + post + resp), "tokenizers must agree"
    it["partners"] = []
    if with_partners:
        rng2 = random.Random(f"{SEED}:{salt}:partner:{depth}:{idx}")
        used_x, tries = {vals[xvar]}, 0
        while len(it["partners"]) < CFG["K_MM"] and tries < 5000:
            tries += 1
            l2, v2, _ = gen_program(rng2, depth)
            p1b = enc(TOK_R, pre + HEADER + "\n".join(l2) + "\n")
            if len(p1b) != len(p1) or v2[xvar] in used_x:                        # exact same length; a different value of the queried variable
                continue
            used_x.add(v2[xvar])
            it["partners"].append(dict(lines=l2, Xp=v2[xvar], g_mm=(v2[xvar] * k + m) % 10, p1=p1b))
        assert len(it["partners"]) == CFG["K_MM"], f"could not find {CFG['K_MM']} same-length wrong programs for {it['id']}"
    return it

def build_items(depths, n_per_depth, salt, with_partners=True):
    items = [make_item(d, i, salt, with_partners) for d in depths for i in range(n_per_depth)]
    random.Random(f"{SEED}:{salt}:order").shuffle(items)       # pre-registered processing order (depths interleaved; truncation would be unbiased)
    return items

def note_ids(it, s_state_digit):
    """T: text handoff. The receiver (alone) also reads A's computed value of the queried variable, placed in the tail before U."""
    note = f"A model that read the program above computed {it['xvar']} = {s_state_digit}. "
    return it["p1"] + enc(TOK_R, note + it["U_text"] + it["post"] + it["resp"])

def arrays(ids, cond, key="a"):
    return np.array([RECORDS[i][cond][key] if RECORDS[i].get(cond) else "" for i in ids])

def _lp_vec(pred):
    """Log-softmax over the four option letters (stored as `lp`; recomputed from `p` if absent)."""
    if pred.get("lp") and pred["lp"][0] is not None:
        return np.array(pred["lp"], float)
    p = np.array(pred["p"], float)
    return np.log(np.clip(p, 1e-12, None))

def cal_folds(ids, seed=None):
    """Deterministic 2-fold split of the scored items; the SAME split is used for every condition (R, C, C_mm)."""
    seed = CFG["CAL_SEED"] if seed is None else seed
    order = sorted(ids)
    perm = list(range(len(order)))
    random.Random(f"{seed}:calfold").shuffle(perm)
    fold = {}
    for rank, j in enumerate(perm):
        fold[order[j]] = rank % 2
    return fold

def crossfit_calibrate(ids, cond, fold=None):
    """PREREG_run2 (a), kept in run 3. Label-free per-letter prior correction, 2-fold cross-fit: items in fold k are scored as
    argmax_l [ logp_l - mean_over_items_of_the_OTHER_fold(logp_l) ] (ties -> lower letter). Uses no labels and no sharer
    information beyond what `cond` itself outputs. Returns {id: letter} for items where `cond` is present and finite."""
    fold = fold if fold is not None else cal_folds(ids)
    have = [i for i in ids if RECORDS[i].get(cond) and RECORDS[i][cond]["a"] != "?"]
    lp = {i: _lp_vec(RECORDS[i][cond]) for i in have}
    out = {}
    for k in (0, 1):
        train = [lp[i] for i in have if fold[i] != k]
        if not train:
            continue
        prior = np.mean(train, axis=0)
        for i in have:
            if fold[i] == k:
                out[i] = LETTERS[int(np.argmax(lp[i] - prior))]
    return out

def cal_array(ids, cond, fold):
    d = crossfit_calibrate(ids, cond, fold)
    return np.array([d.get(i, "?") for i in ids])

def mk_gate_items(n):
    return build_items([CFG["DEPTHS"][0]], n, "gate", with_partners=False)


In [ ]:
# --- 10. Load models (pilot: R and S only; main: + frozen fuser, fp16 gated against an fp32 reference) ---
import contextlib

@contextlib.contextmanager
def on_device(dev):
    global DEVICE
    old = DEVICE
    DEVICE = dev
    try:
        yield
    finally:
        DEVICE = old

def probe_condition_outputs(items, deadline=None):
    out = {"R": [], "S": [], "C": []}
    for it in items:
        if deadline is not None and out["R"] and time.time() > deadline:
            break
        ids = it["ids_full"]
        for k, lg in (("R", lm_logits(RECV, ids)), ("S", lm_logits(SHAR, ids)), ("C", c2c_logits(ids, ids, it["n_p1"]))):
            out[k].append(to_pred(lg))
    return out

def mem_available_gb():
    """Host RAM available to this process in GiB: min(MemAvailable, cgroup limit - cgroup usage). None if unknown."""
    vals = []
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemAvailable:"):
                vals.append(int(line.split()[1]) / 2**20)
    except Exception:
        pass
    for lim_p, cur_p in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                         ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        try:
            lim = int(open(lim_p).read().strip())
            cur = int(open(cur_p).read().strip())
            if lim < 2**50:
                vals.append((lim - cur) / 2**30)
        except Exception:
            continue
    return min(vals) if vals else None

def fp32_fits_gpu():
    if not torch.cuda.is_available():
        return True
    return FP32_GB < 0.80 * torch.cuda.get_device_properties(0).total_memory / 2**30

def load_main():
    """fp16 on a T4, gated against an fp32 reference on probe items (finite; argmax agreement >= 0.80; mean |dprob| <= 0.10 on R, S, C).
    The fp32 stack (~20.6 GB) does not fit the GPU, so the reference runs on the CPU if host RAM allows; else fp16 is accepted on finiteness alone (logged)."""
    force_cpu_ref = _flag("PCS_SMOKE_CPU_REF")
    on_gpu = (fp32_fits_gpu() or SMOKE) and not force_cpu_ref
    ref_dev = DEVICE if on_gpu else torch.device("cpu")
    ref, pool, note = None, None, ""
    try:
        t_ref = time.time()
        if not on_gpu and not SMOKE:
            avail = mem_available_gb()
            print(f"host RAM available: {avail if avail is None else round(avail, 1)} GiB; fp32 reference needs about {FP32_GB} GiB (+3 headroom)", flush=True)
            if avail is not None and avail < FP32_GB + 3:
                raise MemoryError(f"only {avail:.1f} GiB host RAM available")
        with on_device(ref_dev):
            st32 = build_stack(torch.float32, ref_dev); use_stack(st32)
            pool = mk_gate_items(CFG["GATE_PROBES_CPU"])
            ref = probe_condition_outputs(pool, deadline=time.time() + CFG["GATE_CPU_CAP_SEC"])
            free_stack(st32)
        note = f"fp32 reference on {ref_dev.type}: {len(ref['R'])}/{CFG['GATE_PROBES_CPU']} probe items in {time.time()-t_ref:.0f}s"
        if len(ref["R"]) < CFG["GATE_MIN_REF"]:
            note += " (too few: unusable)"; ref = None
    except Exception as e:
        note, ref = f"fp32 reference failed: {repr(e)[:200]}", None
        try: free_stack(st32)
        except Exception: pass
    print(note, flush=True)
    report, ok, st16 = {}, True, None
    try:
        st16 = build_stack(torch.float16); use_stack(st16)
        if pool is None: pool = mk_gate_items(CFG["GATE_PROBES_CPU"])
        got = probe_condition_outputs(pool if ref is None else pool[:len(ref["R"])])
        for k in "RSC":
            fin = all(g["a"] != "?" for g in got[k]); ok &= fin
            if ref is None:
                report[k] = dict(finite=fin); continue
            agree = float(np.mean([g["a"] == r["a"] for g, r in zip(got[k], ref[k])]))
            dp = float(np.mean([np.abs(np.array(g["p"], float) - np.array(r["p"], float)).mean() for g, r in zip(got[k], ref[k]) if g["a"] != "?"])) if fin else float("nan")
            report[k] = dict(finite=fin, argmax_agree=round(agree, 3), mean_abs_dprob=round(dp, 4)); ok &= agree >= 0.80 and dp <= 0.10
    except Exception as e:
        report, ok = {"error": repr(e)[:300]}, False
    print("fp16 gate:", report, flush=True)
    if ok:
        if ref is None:
            DEVIATIONS.append(f"fp32 reference unavailable ({note}); fp16 gated on finiteness only")
        return st16, "fp16 (passed gate vs fp32)" if ref is not None else "fp16 (finite; no fp32 reference)"
    if st16 is not None: free_stack(st16)
    if fp32_fits_gpu() or SMOKE:
        DEVIATIONS.append(f"fp16 failed the gate {report}; fell back to fp32"); return build_stack(torch.float32), "fp32 (fp16 failed gate)"
    DEVIATIONS.append(f"fp16 failed the gate {report}; fp32 does not fit; fell back to bf16 (emulated)")
    return build_stack(torch.bfloat16), "bf16 (fallback)"

t = time.time()
if MODE == "pilot":
    dt = torch.float32 if SMOKE else torch.float16
    STACK, DTYPE_NOTE = build_stack(dt, with_fuser=False), f"{str(dt)} (pilot: no gate, finite-checked)"
else:
    STACK, DTYPE_NOTE = load_main()
use_stack(STACK)
COUNTERS["nonfinite"] = 0
print(f"models ready in {time.time()-t:.0f}s | dtype: {DTYPE_NOTE}")
if torch.cuda.is_available():
    print(f"GPU memory after load: {torch.cuda.memory_allocated()/2**30:.1f} GiB of {torch.cuda.get_device_properties(0).total_memory/2**30:.1f}")
ITEM_LIST = build_items(CFG["DEPTHS"], CFG["N_PER_DEPTH"], "main" if MODE == "main" else "pilot", with_partners=(MODE == "main"))
_L = [it["n_p1"] for it in ITEM_LIST]
print(f"items: {len(ITEM_LIST)} | fused-section tokens median {int(np.median(_L))} (min {min(_L)}, max {max(_L)}) | total tokens median {int(np.median([len(i['ids_full']) for i in ITEM_LIST]))}")
_it = ITEM_LIST[0]
print("example fused section:\n" + _it["fused_text"] + "tail U:", _it["U_text"], "| queried", _it["xvar"], "= X", _it["X"], "-> U-answer", _it["ans"])


In [ ]:
# --- 11. Run: pilot (R, R_state, S_full, S_state) or main (+ C, C_mm x K, T) ---
def save_partial():
    try:
        PARTIAL_PATH.write_text(json.dumps({"records": RECORDS, "config": CFG}))
    except Exception as e:
        print("could not write partial results:", e)

def run_items(items, main):
    t_s = time.time()
    deadline = T0 + 1e9 if not main else time.time() + CFG["MAIN_CAP_MIN"] * 60
    stopped = None
    for n_, it in enumerate(items):
        if time.time() > deadline:
            stopped = f"stopped after {n_} of {len(items)} items at the {CFG['MAIN_CAP_MIN']}-min cap"; break
        ITEMS[it["id"]] = it
        rec = RECORDS.setdefault(it["id"], {})
        rec["R"] = to_pred(lm_logits(RECV, it["ids_full"]))                 # receiver alone on P + U
        rec["R_state"] = to_pred(lm_logits(RECV, it["ids_state"]))          # receiver alone, P-only query for the variable
        rec["S_full"] = to_pred(lm_logits(SHAR, it["ids_full"]))            # A alone on P + U
        rec["S_state"] = to_pred(lm_logits(SHAR, it["ids_state"]))          # A alone, P-only query: "A's belief" about the variable
        if main:
            ids, n_p1 = it["ids_full"], it["n_p1"]
            rec["C"] = to_pred(c2c_logits(ids, ids, n_p1))                  # fused section = P only; U is in the unfused tail, A never sees it
            for k, pt in enumerate(it["partners"]):
                ids_s = pt["p1"] + it["p2U"]
                assert len(ids_s) == len(ids) and ids_s[n_p1:] == ids[n_p1:], "wrong-P cache must have exactly the same length (equal-length assertion)"
                rec[f"C_mm{k}"] = to_pred(c2c_logits(ids, ids_s, n_p1)); rec[f"C_mm{k}"]["g_mm"] = pt["g_mm"]
            if rec["S_state"]["a"] != "?":
                rec["T"] = to_pred(lm_logits(RECV, note_ids(it, rec["S_state"]["a"])))
            if n_ < 40:
                rec.setdefault("_proj_off", to_pred(c2c_logits(ids, ids, n_p1, sharer_mask=-1))["a"])   # projection OFF must reproduce the plain receiver
        if (n_ + 1) % 100 == 0:
            print(f"  {n_+1}/{len(items)}  {time.time()-t_s:.0f}s  nonfinite={COUNTERS['nonfinite']}", flush=True)
            save_partial()
    save_partial()
    return stopped

STOPPED = run_items(ITEM_LIST, MODE == "main")
if STOPPED:
    DEVIATIONS.append(STOPPED)
print("run finished:", STOPPED or "all items processed", f"| {time.time()-T0:.0f}s since start")


In [ ]:
# --- 12. Analysis, reading, save ---
def digit_acc(ids, cond, truth):
    return np.array([RECORDS[i][cond]["a"] == t if RECORDS[i].get(cond) else False for i, t in zip(ids, truth)], float)

def pilot_table():
    rows, out = [], {}
    for d in CFG["DEPTHS"]:
        ids = [i for i in RECORDS if ITEMS[i]["depth"] == d]
        ans = [ITEMS[i]["ans"] for i in ids]; X = [ITEMS[i]["X"] for i in ids]
        fold = cal_folds(ids)
        Rcal = cal_array(ids, "R", fold)
        a = {"R": digit_acc(ids, "R", [str(x) for x in ans]), "S_full": digit_acc(ids, "S_full", [str(x) for x in ans]),
             "S_state": digit_acc(ids, "S_state", [str(x) for x in X]), "R_state": digit_acc(ids, "R_state", [str(x) for x in X]),
             "Rcal": np.array([c == str(t) for c, t in zip(Rcal, ans)], float)}
        ci = {k: boot_ci(v) for k, v in a.items()}
        qual = (d != 0) and (ci["S_state"][0] - ci["R"][0] >= 0.20) and ci["S_state"][0] >= 0.60   # depth 0 is the pipeline sanity check, excluded
        out[d] = {"n": len(ids), "acc": {k: [round(x, 4) for x in v] for k, v in ci.items()}, "S_state_minus_R": round(ci["S_state"][0] - ci["R"][0], 4),
                  "S_state_minus_Rcal": round(ci["S_state"][0] - ci["Rcal"][0] if False else ci["S_state"][0] - ci["Rcal"][0], 4), "qualifies": bool(qual)}
        rows.append([d, len(ids)] + [fmt(ci[k], True) for k in ("R", "Rcal", "R_state", "S_full", "S_state")] + [f"{(ci['S_state'][0]-ci['R'][0])*100:+.1f}", "YES" if qual else ("sanity" if d == 0 else "no")])
    table(rows, ["depth", "n", "R (P+U)", "Rcal (P+U)", "R_state (P only)", "S_full (P+U)", "S_state (P only)", "S_state - R", "qualifies"])
    return out

def analyze_pcs1(ids):
    K = CFG["K_MM"]
    ids = [i for i in ids if all(k in RECORDS[i] for k in ("R", "S_full", "S_state", "C", "T")) and all(f"C_mm{k}" in RECORDS[i] for k in range(K))]
    ans = np.array([str(ITEMS[i]["ans"]) for i in ids]); X = np.array([str(ITEMS[i]["X"]) for i in ids]); depth = np.array([ITEMS[i]["depth"] for i in ids])
    R, Sf, Ss, C, T = (arrays(ids, k) for k in ("R", "S_full", "S_state", "C", "T"))
    Cmk = [arrays(ids, f"C_mm{k}") for k in range(K)]
    fold = cal_folds(ids)
    Rcal, Ccal = cal_array(ids, "R", fold), cal_array(ids, "C", fold)
    Cmcal = [cal_array(ids, f"C_mm{k}", fold) for k in range(K)]
    ok = ~np.isin(R, ["?", ""]) & ~np.isin(Ss, ["?", ""]) & ~np.isin(C, ["?", ""]) & ~np.isin(T, ["?", ""]) & np.logical_and.reduce([~np.isin(a, ["?", ""]) for a in Cmk])
    g = np.array([str((int(s) * ITEMS[i]["k"] + ITEMS[i]["m"]) % 10) if s not in ("?", "") else "?" for s, i in zip(Ss, ids)])   # U-answer implied by A's belief
    gmm = np.array([[ITEMS[i]["partners"][k]["g_mm"] for k in range(K)] for i in ids])                                           # U-answer implied by each wrong cache
    corr = lambda a: (a == ans).astype(float)
    hit = lambda a: (a == g).astype(float)
    W = ok & (Ss != X)                         # A's belief is WRONG
    Aok = ok & (Ss == X)
    mean_cm, mean_cmcal = np.mean([corr(a) for a in Cmk], axis=0), np.mean([corr(a) for a in Cmcal], axis=0)
    mean_hit_cm = np.mean([hit(a) for a in Cmk], axis=0)
    def sub(m):
        return {"n": int(m.sum()),
                "acc": {"R": boot_ci(corr(R)[m]), "Rcal": boot_ci(corr(Rcal)[m]), "S_full": boot_ci(corr(Sf)[m]), "S_state": boot_ci((Ss == X).astype(float)[m]), "C": boot_ci(corr(C)[m]),
                        "C_mm_mean": boot_ci(mean_cm[m]), "T": boot_ci(corr(T)[m]), "Ccal": boot_ci(corr(Ccal)[m])},
                "Delta_specific": boot_ci((corr(C) - mean_cm)[m]), "C_minus_Rcal": boot_ci((corr(C) - corr(Rcal))[m]),
                "Delta_specific_cal": boot_ci((corr(Ccal) - mean_cmcal)[m]), "C_minus_R": boot_ci((corr(C) - corr(R))[m]), "C_minus_T": boot_ci((corr(C) - corr(T))[m]),
                "T_minus_Rcal": boot_ci((corr(T) - corr(Rcal))[m]), "C_minus_Sfull": boot_ci((corr(C) - corr(Sf))[m])}
    out = {"K": K, "n_items": int(ok.sum()), "n_A_wrong_belief": int(W.sum()), "n_A_right_belief": int(Aok.sum()), "depths": CFG["DEPTHS"],
           "pooled": sub(ok), "by_depth": {int(d): sub(ok & (depth == d)) for d in sorted(set(depth.tolist()))}}
    fid = {"n": int(W.sum()), "chance_reference": 0.1,
           "P_C_eq_g": boot_ci(hit(C)[W]), "P_Cmm_eq_g_mean": boot_ci(mean_hit_cm[W]), "excess": boot_ci((hit(C) - mean_hit_cm)[W]),
           "P_C_eq_g_minus_chance": boot_ci(hit(C)[W] - 0.1), "P_R_eq_g": boot_ci(hit(R)[W]), "P_Rcal_eq_g": boot_ci(hit(Rcal)[W]), "P_T_eq_g": boot_ci(hit(T)[W]),
           "excess_vs_Rcal": boot_ci((hit(C) - hit(Rcal))[W]),
           "per_k_Cmm_eq_g": [boot_ci(hit(a)[W]) for a in Cmk],
           "by_depth": {int(d): boot_ci((hit(C) - mean_hit_cm)[W & (depth == d)]) for d in sorted(set(depth.tolist()))}}
    out["fidelity"] = fid
    out["fidelity_all_items_agree_with_A_implied_answer"] = {"P_C_eq_g": boot_ci(hit(C)[ok]), "P_Cmm_eq_g_mean": boot_ci(mean_hit_cm[ok]), "excess": boot_ci((hit(C) - mean_hit_cm)[ok])}
    out["wrong_cache_following"] = {"P_Cmm_eq_g_mm_mean": boot_ci(np.mean([(Cmk[k] == gmm[:, k].astype(str)).astype(float) for k in range(K)], axis=0)[ok]),
                                    "P_C_eq_g_mm_mean": boot_ci(np.mean([(C == gmm[:, k].astype(str)).astype(float) for k in range(K)], axis=0)[ok])}
    out["agreement_C_vs_Cmm"] = [boot_ci((C == a)[ok]) for a in Cmk]
    out["agreement_C_vs_R"] = boot_ci((C == R)[ok])
    out["proj_off_matches_receiver"] = None
    po = [(RECORDS[i]["_proj_off"], RECORDS[i]["R"]["a"]) for i in ids if "_proj_off" in RECORDS[i]]
    if po: out["proj_off_matches_receiver"] = float(np.mean([a == b for a, b in po]))
    out["digit_marginals"] = {k: {l: round(float((a[ok] == l).mean()), 3) for l in LETTERS} for k, a in (("truth_U", ans), ("R", R), ("Rcal", Rcal), ("C", C), ("C_mm0", Cmk[0]), ("T", T))}
    out["reading"], out["reading_flags"] = reading_pcs1(out)
    return out

def reading_pcs1(o):
    """PREREG_pcs1 decision rule (pooled over the frozen depths)."""
    P = o["pooled"]; d, dc, f = P["Delta_specific"], P["C_minus_Rcal"], o["fidelity"]["excess"]
    if any(x != x for x in (d[0], dc[0], f[0])):
        return "NO CLAIM: missing data", {}
    ded = bool(d[1] >= 0.02 and dc[1] > 0)
    fid = bool(f[1] > 0)
    null = bool(d[1] <= 0 <= d[2] and f[1] <= 0 <= f[2])
    flags = {"new_deduction_supported(Delta lowerCI>=+0.02 and C-Rcal lowerCI>0)": ded, "fidelity_supported(excess lowerCI>0)": fid, "both_CIs_include_0": null}
    mag = f"Delta_specific {d[0]:+.3f} [{d[1]:+.3f},{d[2]:+.3f}]; C-Rcal {dc[0]:+.3f} [{dc[1]:+.3f},{dc[2]:+.3f}]; fidelity excess {f[0]:+.3f} [{f[1]:+.3f},{f[2]:+.3f}]"
    if ded and fid: return f"NEW DEDUCTION SUPPORTED and FIDELITY SUPPORTED. {mag}", flags
    if ded: return f"NEW DEDUCTION SUPPORTED (fidelity not supported). {mag}", flags
    if fid: return f"FIDELITY SUPPORTED (new deduction not supported). {mag}", flags
    if null: return f"NULL: both CIs include 0. {mag}", flags
    return f"PARTIAL. {mag}", flags

def print_pcs1(o):
    P = o["pooled"]
    print(f"\n=== PCS1 main: depths {o['depths']}, items scored {o['n_items']} (A's belief wrong on {o['n_A_wrong_belief']}) ===")
    table([[k, fmt(v, True)] for k, v in P["acc"].items()], ["pooled U-answer accuracy (chance 10%)", "[95% CI]"])
    table([[k, fmt(P[k])] for k in ("Delta_specific", "Delta_specific_cal", "C_minus_Rcal", "C_minus_R", "C_minus_T", "T_minus_Rcal", "C_minus_Sfull")], ["paired difference", "[95% CI]"])
    for d, s in o["by_depth"].items():
        print(f"  depth {d}: n={s['n']} | " + " | ".join(f"{k} {fmt(v, True)}" for k, v in s["acc"].items()) + f" | Delta {fmt(s['Delta_specific'])} | C-Rcal {fmt(s['C_minus_Rcal'])}")
    f = o["fidelity"]
    print(f"\nFIDELITY on items where A's belief is wrong (n={f['n']}); g = U-answer implied by A's wrong belief; chance 0.10")
    table([["P(C == g)", fmt(f["P_C_eq_g"], True)], ["P(C_mm == g), mean over k", fmt(f["P_Cmm_eq_g_mean"], True)], ["EXCESS P(C==g) - mean P(C_mm,k==g)", fmt(f["excess"])],
           ["P(C == g) - 0.10", fmt(f["P_C_eq_g_minus_chance"])], ["P(R == g)", fmt(f["P_R_eq_g"], True)], ["P(Rcal == g)", fmt(f["P_Rcal_eq_g"], True)], ["P(T == g) text handoff", fmt(f["P_T_eq_g"], True)]], ["quantity", "[95% CI]"])
    print("by depth excess:", {d: fmt(v) for d, v in f["by_depth"].items()})
    print("wrong-cache following:", {k: fmt(v, True) for k, v in o["wrong_cache_following"].items()})
    print("agreement C vs C_mm,k:", [fmt(a, True) for a in o["agreement_C_vs_Cmm"]], "| C vs R:", fmt(o["agreement_C_vs_R"], True), "| projection-off == receiver:", o["proj_off_matches_receiver"])
    print("digit marginals:", o["digit_marginals"])
    print("\nREADING (PREREG_pcs1):", o["reading"], o["reading_flags"])

RES = {"config": CFG, "mode": MODE, "deviations": DEVIATIONS, "run": {"date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "gpu": GPU_NAME, "dtype_note": DTYPE_NOTE,
       "torch": torch.__version__, "transformers": installed("transformers"), "c2c_commit": head, "receiver": RECV_ID, "sharer": SHAR_ID, "smoke": SMOKE,
       "wall_seconds": round(time.time() - T0), "nonfinite_logits": COUNTERS["nonfinite"]}}
if MODE == "pilot":
    RES["pilot"] = pilot_table()
    print("\nQUALIFYING depths (S_state - R >= 20 pts and S_state >= 60%):", [d for d, v in RES["pilot"].items() if v["qualifies"]] or "NONE")
else:
    try:
        RES["pcs1"] = analyze_pcs1(list(RECORDS))
        print_pcs1(RES["pcs1"])
    except Exception as e:
        import traceback; traceback.print_exc()
        DEVIATIONS.append(f"analysis raised {repr(e)[:200]}; records are saved; re-run offline")
for it_id, rec in RECORDS.items():
    it = ITEMS[it_id]
    rec["_meta"] = {k: it[k] for k in ("depth", "xvar", "X", "k", "m", "ans")}
RES["records"] = RECORDS
def jsonable(o):
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, tuple): return list(o)
    raise TypeError(type(o))
try:
    RESULTS_PATH.write_text(json.dumps(RES, default=jsonable))
except Exception as e:
    print("strict dump failed:", repr(e)[:200]); RESULTS_PATH.write_text(json.dumps(RES, default=str))
print(f"\nsaved {RESULTS_PATH} ({RESULTS_PATH.stat().st_size/1e6:.1f} MB). Total wall time {(time.time()-T0)/60:.1f} min.")
if DEVIATIONS:
    print("DEVIATIONS:", *DEVIATIONS, sep="\n  - ")
